In [ ]:
import spacy
import os
import random
import pandas as pd
from collections import defaultdict
import stanza
import re


In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["ner"])

## Code to join noun phrases within sentences
For the NP shuffling you first do a complete pass through the data with the join_noun_phrases and then you apply the same shuffle_1gram algorithm.

In [2]:
def join_noun_phrases_1(sentence):
    doc = nlp(sentence)

    # Collect all noun chunk spans, sorted by start index
    noun_chunks = list(doc.noun_chunks)
    noun_chunks.sort(key=lambda np: np.start)

    reconstructed = []
    i = 0
    while i < len(doc):
        # Check if this token starts a noun chunk
        match = None
        for np in noun_chunks:
            if np.start == i:
                match = np
                break

        if match:
            # Join NP tokens with underscores, keep the whitespace of the last token
            np_text = "_".join([t.text for t in match])
            trailing_ws = match[-1].whitespace_
            reconstructed.append(np_text + trailing_ws)
            i = match.end
        else:
            reconstructed.append(doc[i].text_with_ws)
            i += 1

    return "".join(reconstructed).strip()

## CHILDES

In [ ]:
with open('./corpora/CHILDES_rand/original/childes_rand.train.txt', 'r') as file:
    with open('./corpora/CHILDES_rand/shuffle_order/np/childes_rand_np_train.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_np = join_noun_phrases_1(line)
            if line_np is not None:
                outfile.write(line_np + '\n')

In [ ]:
with open('./corpora/CHILDES_rand/original/childes_rand.dev.txt', 'r') as file:
    with open('./corpora/CHILDES_rand/shuffle_order/np/childes_rand_np_dev.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_np = join_noun_phrases_1(line)
            outfile.write(line_np + '\n')

## WIKIPEDIA

In [ ]:
with open('./corpora/WIKI_rand/original/wiki_rand.train.txt', 'r') as file:
    with open('./corpora/WIKI_rand/shuffle_order/np/wiki_rand_np_train.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_np = join_noun_phrases_1(line.strip())
            outfile.write(line_np + '\n')
            

In [ ]:
with open('./corpora/WIKI_rand/original/wiki_rand.dev.txt', 'r') as file:
    with open('./corpora/WIKI_rand/shuffle_order/np/wiki_rand_np_dev.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_np = join_noun_phrases_1(line.strip())
            outfile.write(line_np + '\n')

## BNC

In [ ]:
with open('./corpora/BNC_rand/original/bnc_rand.train.txt', 'r') as file:
    with open('./corpora/BNC_rand/shuffle_order/np/bnc_rand_np_train.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_np = join_noun_phrases_1(line.strip())
            outfile.write(line_np + '\n')

In [ ]:
with open('./corpora/BNC_rand/original/bnc_rand.dev.txt', 'r') as file:
    with open('./corpora/BNC_rand/shuffle_order/np/bnc_rand_np_dev.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_np = join_noun_phrases_1(line.strip())
            outfile.write(line_np + '\n')

## CANDOR

In [ ]:
with open('./corpora/CANDOR_rand/original/candor_rand.train.txt', 'r') as file:
    with open('./corpora/CANDOR_rand/shuffle_order/np/candor_noun_phrases_train.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_np = join_noun_phrases_1(line.strip())
            outfile.write(line_np + '\n')

In [ ]:
with open('./corpora/CANDOR_rand/original/candor_rand.dev.txt', 'r') as file:
    with open('./corpora/CANDOR_rand/shuffle_order/np/candor_noun_phrases_dev.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_np = join_noun_phrases_1(line.strip())
            outfile.write(line_np + '\n')

## SHUFFLE 1 GRAM

In [ ]:
def shuffle_1gram(sentence, seed=None):
    """
    Shuffle a sentence at the 1-gram (token) level while keeping punctuation (except apostrophes) in place.
    Ensures that no non-punctuation token remains in its original position.
    
    Args:
        sentence (str): Input sentence.
        seed (int, optional): Random seed for reproducibility.

    Returns:
        str: Shuffled sentence with punctuation restored.
    """
    if seed is not None:
        random.seed(seed)

    # Regex: words including internal apostrophes OR any punctuation except apostrophes
    token_pattern = re.compile(r"\b\w+(?:'\w+)?(?:_\w+)*\b|[^\w\s']")
    tokens = token_pattern.findall(sentence)

    # Store punctuation positions (except apostrophes inside words)
    punct_positions = {i: tok for i, tok in enumerate(tokens) if re.match(r"[^\w\s']", tok)}

    # Extract only non-punctuation tokens for shuffling
    words = [tok for i, tok in enumerate(tokens) if i not in punct_positions]

    # Edge case: if 1 or 0 words, return original
    if len(words) <= 1:
        return sentence

    # Shuffle until no word remains in its original position
    attempts = 0
    while True:
        shuffled_words = words.copy()
        random.shuffle(shuffled_words)
        same_position_count = sum([w1 == w2 for w1, w2 in zip(words, shuffled_words)])
        attempts += 1
        if same_position_count == 0 or attempts > 100:
            break

    # Reconstruct sentence with punctuation in original positions
    shuffled_tokens = []
    word_idx = 0
    for i in range(len(tokens)):
        if i in punct_positions:
            shuffled_tokens.append(punct_positions[i])
        else:
            shuffled_tokens.append(shuffled_words[word_idx])
            word_idx += 1

    shuffles_tr = " ".join(shuffled_tokens)
    #check if there is a space before the punctutattion at the very end of the sentence and remove it
    if re.search(r"\s[.,!?;:]$", shuffles_tr):
        shuffles_tr = re.sub(r"\s([.,!?;:])$", r"\1", shuffles_tr)
    return shuffles_tr



## CHILDES

In [ ]:
with open('./corpora/CHILDES_rand/shuffle_order/np/childes_rand_np_train.txt', 'r') as file:
    with open('./corpora/CHILDES_rand/shuffle_order/np/childes_rand_np_train_scrambled.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')


with open('./corpora/CHILDES_rand/shuffle_order/np/childes_rand_np_dev.txt', 'r') as file:
    with open('./corpora/CHILDES_rand/shuffle_order/np/childes_rand_np_dev_scrambled.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')

## WIKIPEDIA

In [ ]:
with open('./corpora/WIKI_rand/shuffle_order/np/wiki_rand_np_train.txt', 'r') as file:
    with open('./corpora/WIKI_rand/shuffle_order/np/wiki_rand_np_train_scrambled.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')
    

with open('./corpora/WIKI_rand/shuffle_order/np/wiki_rand_np_dev.txt', 'r') as file:
    with open('./corpora/WIKI_rand/shuffle_order/np/wiki_rand_np_dev_scrambled', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')


## BNC


In [ ]:
with open('./corpora/BNC_rand/shuffle_order/np/bnc_rand_np_train.txt', 'r') as file:
    with open('./corpora/BNC_rand/shuffle_order/np/bnc_rand_np_train_scrambled.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')

with open('./corpora/BNC_rand/shuffle_order/np/bnc_rand_np_dev.txt', 'r') as file:
    with open('./corpora/BNC_rand/shuffle_order/np/bnc_rand_np_dev_scambled.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')

## CANDOR

In [ ]:
with open('./corpora/CANDOR_rand/shuffle_order/np/candor_noun_phrases_train.txt', 'r') as file:
    with open('./corpora/CANDOR_rand/shuffle_order/np/candor_noun_phrases_train_scrambled.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')
    

with open('./corpora/CANDOR_rand/shuffle_order/np/candor_noun_phrases_dev.txt', 'r') as file:
    with open('./corpora/CANDOR_rand/shuffle_order/np/candor_noun_phrases_dev_scrambled.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')

## Decode Noun Phrases correctly

In [7]:
# Regex patterns for post-processing
punct_fix_end = re.compile(r"\s+([.,!?;:])$")       # Remove space before ending punctuation
ellipsis_patterns = [
    re.compile(r"\s*\.\s*\.\s*\.\s*"),             # any spaced ... sequences
    re.compile(r"\s*\.\s*\.\s*"),                  # .. sequences
]
quotes_fix = re.compile(r"[“”]\s*(.*?)\s*[“”]", flags=re.UNICODE)  # text between quotes
hyphen_fix = re.compile(r"\b([A-Za-z]+)\s*-\s*([A-Za-z]+)\b")   
hyphen_fix3 = re.compile(r"\b([A-Za-z]+)\s*-\s*([A-Za-z]+)\s*-\s*([A-Za-z]+)\b")     # join split hyphenated words
hyphen_fix4 = re.compile(r"\b([A-Za-z]+)\s*-\s*([A-Za-z]+)\s*-\s*([A-Za-z]+)\s*-\s*([A-Za-z]+)\b") 
hyphen_fix5 = re.compile(r"\b([A-Za-z]+)\s*-\s*([A-Za-z]+)\s*-\s*([A-Za-z]+)\s*-\s*([A-Za-z]+)\s*-\s*([A-Za-z]+)\b") 
clitic_fix = re.compile(r"\b([A-Za-z]+)\s+('re|'m|'s|'ve|'d|'ll|n['’]t)\b", flags=re.IGNORECASE)
INSIDE_BRACKETS = re.compile(r"([\(\[\{])\s*(.*?)\s*([\)\]\}])")
INSIDE_QUOTES = re.compile(r"([\"'‘“”])\s*(.*?)\s*([\"'“’”])")
REMOVE_SPACE_BEFORE_PUNCT = re.compile(r"\s+([,.:;!?])") 
hyphen_fix_pattern = re.compile(r"\b([A-Za-z0-9]+)\s*-\s*([A-Za-z0-9]+)\b")


def normalize_sentence(text):
    # Preserve leading spaces
    leading_space = ""
    if text.startswith(" "):
        leading_space = " "
    
    text = text.strip()

    # Fix clitics (general contractions)
    text = clitic_fix.sub(lambda m: m.group(1) + m.group(2), text)

    # Fix hyphenated words: join split hyphenated words
    text = hyphen_fix.sub(r"\1-\2", text)
    text = hyphen_fix3.sub(r"\1-\2-\3", text)
    text = hyphen_fix4.sub(r"\1-\2-\3-\4", text)
    text = hyphen_fix5.sub(r"\1-\2-\3-\4-\5", text)

    # Fix text inside quotes: “ text ” -> “text”
    text = quotes_fix.sub(r'“\1”', text)
    text = re.sub(r"\s+([.,!?;:])\s*", r"\1 ", text)
    text = re.sub(r"([.,!?;:])\s+\"", r'\1"', text)

    # Fix ellipses
    for pattern in ellipsis_patterns:
        text = pattern.sub("...", text)

    # Remove spaces before ending punctuation
    text = punct_fix_end.sub(r"\1", text)
    text = REMOVE_SPACE_BEFORE_PUNCT.sub(r"\1", text)
    text = INSIDE_BRACKETS.sub(r"\1\2\3", text)
    text = INSIDE_QUOTES.sub(r"\1\2\3", text)
    text = hyphen_fix_pattern.sub(r"\1-\2", text)

    if text.startswith("' "):
        text = text.replace("' ", "'", 1)
    if text.startswith('" '):
        text = text.replace('" ', '"', 1)
    if text.endswith('  . "'):
        text = text.replace('  . "', '."', 1)
    if ' ...' in text:
        text = text.replace(' ...', '...', 1)
    if ' . ..' in text:
        text = text.replace(' . ..', '...', 1)
    if ' . . .' in text:
        text = text.replace(' . . .', '...', 1)

    # Collapse multiple spaces inside sentence
    text = re.sub(r"\s+", " ", text).strip()

    return leading_space + text


def decode_noun_phrases(text):
    text = text.replace("_", " ")
    # Remove spaces right after an opening bracket
    text = re.sub(r"([\(\[\{])\s+", r"\1", text)
    # Remove spaces right before a closing bracket or punctuation
    text = re.sub(r"\s+([\)\]\}\.,!?;:])", r"\1", text)
    text = normalize_sentence(text)
    return text

CHILDES RAND

In [ ]:
with open('./corpora/CHILDES_rand/shuffle_order/np/childes_rand_np_train_scrambled.txt', 'r') as file:
    with open('./corpora/CHILDES_rand/shuffle_order/np_final/childes_rand.train.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_decoded = decode_noun_phrases(line.strip())
            outfile.write(line_decoded + '\n')

In [ ]:
with open('./corpora/CHILDES_rand/shuffle_order/np/childes_rand_np_dev_scrambled.txt', 'r') as file:
    with open('./corpora/CHILDES_rand/shuffle_order/np_final/childes_rand.dev.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_decoded = decode_noun_phrases(line.strip())
            outfile.write(line_decoded + '\n')

WIKI RAND

In [ ]:
with open('./corpora/WIKI_rand/shuffle_order/np/wiki_rand_np_train_scrambled.txt', 'r') as file:
    with open('./corpora/WIKI_rand/shuffle_order/np_final/wiki_rand.train.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_decoded = decode_noun_phrases(line.strip())
            outfile.write(line_decoded + '\n')

with open('./corpora/WIKI_rand/shuffle_order/np/wiki_rand_np_dev_scrambled.txt', 'r') as file:
    with open('./corpora/WIKI_rand/shuffle_order/np_final/wiki_rand.dev.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_decoded = decode_noun_phrases(line.strip())
            outfile.write(line_decoded + '\n')

BNC RAND

In [ ]:
with open('./corpora/BNC_rand/shuffle_order/np/bnc_rand_np_train_scrambled.txt', 'r') as file:
    with open('./corpora/BNC_rand/shuffle_order/np_final/bnc_rand.train.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_decoded = decode_noun_phrases(line.strip())
            outfile.write(line_decoded + '\n')

with open('./corpora/BNC_rand/shuffle_order/np/bnc_rand_np_dev_scambled.txt', 'r') as file:
    with open('./corpora/BNC_rand/shuffle_order/np_final/bnc_rand.dev.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_decoded = decode_noun_phrases(line.strip())
            outfile.write(line_decoded + '\n')

CANDOR

In [8]:
with open('./corpora/CANDOR_rand/shuffle_order/np/candor_noun_phrases_train_scrambled.txt', 'r') as file:
    with open('./corpora/CANDOR_rand/shuffle_order/np_final/candor.train.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_decoded = decode_noun_phrases(line.strip())
            outfile.write(line_decoded + '\n')

with open('./corpora/CANDOR_rand/shuffle_order/np/candor_noun_phrases_dev_scrambled.txt', 'r') as file:
    with open('./corpora/CANDOR_rand/shuffle_order/np_final/candor.dev.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            line_decoded = decode_noun_phrases(line.strip())
            outfile.write(line_decoded + '\n')